In [2]:
import os
from pathlib import Path
from uuid import uuid4
from qdrant_client import QdrantClient, models
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from __future__ import annotations
from dotenv import load_dotenv
load_dotenv()

True

## Configurations

In [ ]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_CLUSTER_ENDPOINT = os.getenv("QDRANT_CLUSTER_ENDPOINT")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing")

if not QDRANT_API_KEY:
    raise ValueError("QDRANT_API_KEY is missing")

if not QDRANT_CLUSTER_ENDPOINT:
    raise ValueError("QDRANT_CLUSTER_ENDPOINT is missing")

In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

dimension = len(
    embeddings.embed_query("dimension check")
)
print("Embedding dimension:", dimension)

Embedding dimension: 3072


In [5]:
model = ChatOpenAI(model="gpt-4o-mini")

## Setting Qdrant

In [6]:
COLLECTION_NAME = "research-paper-rag"

In [9]:
client = QdrantClient(api_key=QDRANT_API_KEY, url=QDRANT_CLUSTER_ENDPOINT)

In [10]:
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(
            size=dimension,
            distance=models.Distance.COSINE,
        ),
    )

## Loading PDF

In [11]:
loader = PyPDFLoader("data\\llama2-research-paper.pdf")
pages = loader.load()

print("PDF pages loaded:", len(pages))

Exceeded 5000 form XObject invocations while extracting text; further form content is skipped.


PDF pages loaded: 77


## Splitting Documents

In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=120,
)

chunks = text_splitter.split_documents(pages)
print("Chunks created:", len(chunks))

Chunks created: 174


## Adding Metadata

In [13]:
for chunk_index, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_index
    chunk.metadata["filename"] = "llama2-research-paper.pdf"

## Creating Vector Store

In [14]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

In [15]:
#### Use deterministic or stable IDs in production ####
chunk_ids = [
    str(uuid4())
    for _ in chunks
]

## Adding Documents

In [16]:
inserted_ids = vector_store.add_documents(
    documents=chunks,
    ids=chunk_ids,
)

print("Inserted chunks:", len(inserted_ids))

Inserted chunks: 174


## Similarity Search

In [17]:
query = "What is the total training time of Llama2"

results = vector_store.similarity_search(
    query=query,
    k=4,
)

for rank, document in enumerate(results, start=1):
    print(f"\nResult {rank}")
    print("Content:", document.page_content)
    print("Metadata:", document.metadata)


Result 1
Content: 2.2 Training Details
We adopt most of the pretraining setting and model architecture fromLlama 1. We use the standard
transformer architecture (Vaswani et al., 2017), apply pre-normalization using RMSNorm (Zhang and
Sennrich, 2019), use the SwiGLU activation function (Shazeer, 2020), and rotary positional embeddings
(RoPE, Su et al. 2022). The primary architectural differences fromLlama 1 include increased context length
and grouped-query attention (GQA). We detail in Appendix Section A.2.1 each of these differences with
ablation experiments to demonstrate their importance.
Hyperparameters. We trained using the AdamW optimizer (Loshchilov and Hutter, 2017), withβ1 =
0.9,β 2 = 0.95,eps = 10−5. We use a cosine learning rate schedule, with warmup of 2000 steps, and decay
final learning rate down to 10% of the peak learning rate. We use a weight decay of0.1and gradient clipping
of 1.0. Figure 5 (a) shows the training loss forLlama 2 with these hyperparameters.
5
Metadata

## Creating Retriever

In [18]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 4
    },
)

context_documents = retriever.invoke(
    "What are the different variants of Llama2 Chat model"
)

context = "\n\n".join(
    document.page_content
    for document in context_documents
)

print(context)

Figure 3: Safety human evaluation results forLlama 2-Chat compared to other open-source and closed-
source models. Human raters judged model generations for safety violations across ~2,000 adversarial
prompts consisting of both single and multi-turn prompts. More details can be found in Section 4.4. It is
important to caveat these safety results with the inherent bias of LLM evaluations due to limitations of the
prompt set, subjectivity of the review guidelines, and subjectivity of individual raters. Additionally, these
safety evaluations are performed using content standards that are likely to be biased towards theLlama
2-Chatmodels.
We are releasing the following models to the general public for research and commercial use‡:
1. Llama 2, an updated version ofLlama 1, trained on a new mix of publicly available data. We also
increased the size of the pretraining corpus by 40%, doubled the context length of the model, and
adopted grouped-query attention (Ainslie et al., 2023). We are rel